# Эмпирические функции изображений
Данные скрипты выводят гистограммы и графики эмпирической функции распределения как для исходных, так и для эквализованных изображений.

In [ ]:

import numpy as np;
import cv2 as cv;
import matplotlib.pyplot as plt;


def makeCdf(histogram):
    integral = np.zeros_like(histogram);
    integral[0] = histogram[0];
    for i in range(1, integral.shape[0]):
        integral[i] = integral[i - 1] + histogram[i];
    return integral;


images = ['img1.jpg', 'img2.jpg', 'img3.jpg'];
labels = ['Исходное изображение', 'Недоэкспонированное изображение',
          'Переэкспонированное изображение'];

for image_name, label in zip(images, labels):
    img_bgr = cv.imread(image_name);
    gray = cv.cvtColor(img_bgr, cv.COLOR_BGR2GRAY);

    # Эквализация с помощью OpenCV
    equalized_cv = cv.equalizeHist(gray);

    bins = [i for i in range(256)];

    # Гистограмма исходного изображения
    h_original = cv.calcHist([gray], [0], None, [256], [0, 256]);

    # Гистограмма эквализованного изображения
    h_cv = cv.calcHist([equalized_cv], [0], None, [256], [0, 256]);

    fig, ax = plt.subplots(2, 2, figsize=(12, 8));

    # Исходное изображение
    ax[0, 0].imshow(gray, cmap='gray', vmin=0, vmax=255);
    ax[0, 0].set_title('Исходное изображение');
    ax[0, 0].axis('off');

    # Гистограмма и CDF исходного изображения
    ax[0, 1].bar(bins, h_original.flatten(), color='black');
    cdf_original = makeCdf(h_original).flatten();
    cdf_ax = ax[0, 1].twinx();
    cdf_ax.plot(bins, cdf_original, color='red');
    cdf_ax.set_ylim(0, cdf_original[-1]);
    cdf_ax.set_ylabel(
        "Количество пикселей, которые имеют заданное\n"
        "или меньшее значение оттенка"
    );
    ax[0, 1].set_title('Гистограмма исходного изображения');

    # Эквализованное изображение
    ax[1, 0].imshow(equalized_cv, cmap='gray', vmin=0, vmax=255);
    ax[1, 0].set_title('cv.equalizeHist()');
    ax[1, 0].axis('off');

    # Гистограмма и CDF эквализованного изображения
    ax[1, 1].bar(bins, h_cv.flatten(), color='black');
    cdf_cv = makeCdf(h_cv).flatten();
    cdf_ax = ax[1, 1].twinx();
    cdf_ax.plot(bins, cdf_cv, color='blue');
    cdf_ax.set_ylim(0, cdf_cv[-1]);
    cdf_ax.set_ylabel(
        "Количество пикселей, которые имеют заданное\n"
        "или меньшее значение оттенка"
    );
    ax[1, 1].set_title('Гистограмма после cv.equalizeHist()');

    fig.suptitle(label);
    fig.tight_layout();
    plt.show();
